## 0 - Processing de Donnees


In [9]:
import os
import numpy as np
import pandas as pd

RAW_DIR = os.path.join("..","data", "raw")
PROCESSED_DIR = os.path.join("..","data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

DATA_PATH = os.path.join(RAW_DIR, "diabetic_data.csv")
MAPPING_PATH = os.path.join(RAW_DIR, "IDs_mapping.csv")

df = pd.read_csv(DATA_PATH)
ids_map = pd.read_csv(MAPPING_PATH)

print("RAW shape:", df.shape)
print(df.head(3))


RAW shape: (101766, 50)
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No                   No   
1                 3  ...          No      Up                   No   
2                 2  ...          No      No                   No   

   glipizide-metformin  glimepiride-pioglitazone  metformin-rosiglitazone  \
0                   No                      

## 1) Standardiser les valeurs manquantes
 Dans ce dataset, `"?"` représente souvent un manquant.

In [10]:
df = df.replace("?", np.nan)

## 2) Nettoyage de la cible `readmitted` (multiclasse)
 On garde uniquement `<30`, `>30`, `NO`.

In [11]:
TARGET = "readmitted"
allowed = {"<30", ">30", "NO"}

if TARGET not in df.columns:
    raise ValueError(f"Colonne cible introuvable: {TARGET}")

df[TARGET] = df[TARGET].astype(str).str.strip()
df = df[df[TARGET].isin(allowed)].copy()

print("Target distribution:")
print(df[TARGET].value_counts(dropna=False))

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


## 3) Enrichissement diagnostique (2eme source) : mapping diag → catégorie clinique
 On crée:
 - diag_1_cat, diag_2_cat, diag_3_cat
 - et un référentiel `diag_category.csv` (la 2eme source)

In [ ]:
def diag_to_category(code: str) -> str:
    if pd.isna(code):
        return "UNKNOWN"
    code = str(code).strip()
    if code == "":
        return "UNKNOWN"

    if code.startswith("V"):
        return "SUPPORT_CARE_V"
    if code.startswith("E"):
        return "INJURY_E"

    try:
        val = float(code)
    except Exception:
        return "OTHER"

    # Nomenclature d'apres ICD-9
    if 250 <= val < 251:
        return "DIABETES"
    if 390 <= val <= 459 or val == 785:
        return "CARDIO"
    if 460 <= val <= 519 or val == 786:
        return "RESPIRATORY"
    if 520 <= val <= 579 or val == 787:
        return "DIGESTIVE"
    if 580 <= val <= 629:
        return "GENITOURINARY"
    if 800 <= val <= 999:
        return "INJURY"
    return "OTHER"


diag_cols = [c for c in ["diag_1", "diag_2", "diag_3"] if c in df.columns]
for c in diag_cols:
    df[f"{c}_cat"] = df[c].map(diag_to_category)

# Construction du referentiel
all_cats = pd.unique(pd.concat([df[f"{c}_cat"]
                     for c in diag_cols], ignore_index=True))
diag_category = pd.DataFrame({
    "diag_category": sorted([x for x in all_cats if pd.notna(x)])
})
diag_category_path = os.path.join(PROCESSED_DIR, "diag_category.csv")
diag_category.to_csv(diag_category_path, index=False)

print("Saved:", diag_category_path)
print(diag_category)

Saved: ..\data\processed\diag_category.csv
    diag_category description
0          CARDIO            
1        DIABETES            
2       DIGESTIVE            
3   GENITOURINARY            
4          INJURY            
5        INJURY_E            
6           OTHER            
7     RESPIRATORY            
8  SUPPORT_CARE_V            
9         UNKNOWN            


## 4) Export du dataset processed

In [14]:
out_path = os.path.join(PROCESSED_DIR, "dataset_model.csv")
df_model.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Processed shape:", df_model.shape)

Saved: ..\data\processed\dataset_model.csv
Processed shape: (101766, 53)


## 5) Checks d'exigences

In [15]:
assert df_model.shape[0] > 50000, "Le volume minimum 50k lignes n'est pas atteint."
print("OK volume > 50k.")

print(df_model[TARGET].value_counts(normalize=True))
print("NA rates (top 10):")
print(df_model.isna().mean().sort_values(ascending=False).head(10))

OK volume > 50k.
readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64
NA rates (top 10):
weight               0.968585
max_glu_serum        0.947468
A1Cresult            0.832773
medical_specialty    0.490822
payer_code           0.395574
race                 0.022336
diag_3               0.013983
diag_2               0.003518
diag_1               0.000206
age                  0.000000
dtype: float64
